In [58]:
import numpy as np
import xarray as xr
import pandas as pd
import re
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from metpy.plots import colortables
import os

In [59]:
slgt = False
if slgt:
    slgt_str = '_slgt'
else:
    slgt_str = ''

In [60]:
# split days into positive/negative/near zero bias (for one of 12 bias types)--can be done with just targets!
targets = xr.open_dataset(f'data/processed_data/train_targets{slgt_str}.nc')

In [78]:
sd_split = .431

variables = targets.variable.values

# Preallocate the output variable directly (same shape/coords as desired)
day_ds = xr.Dataset(
    {
        "bias_sign": (
            ("hazard", "variable", "time"),
            np.empty((len(targets.hazard), len(variables), len(targets.time)), dtype=object),
        )
    },
    coords={
        "hazard": targets.hazard,
        "variable": variables,
        "time": targets.time,
    },
)

# Fill bias_sign for each variable
for variable in variables:
    vals = targets[variable]  # shape: (hazard, time)
    day_ds["bias_sign"].loc[dict(variable=variable)] = xr.where(
        vals > sd_split,
        "pos",
        xr.where(vals < -sd_split, "neg", "zero"),
    )


In [79]:
day_ds.to_netcdf(f"data/processed_data/day_splits{slgt_str}.nc")

Making Plots

In [61]:
day_ds = xr.open_dataset(f"data/processed_data/day_splits{slgt_str}.nc")

In [85]:
# plot composite (uncentered) weather variable and outlook of given set of days function
# plot composite (centered) weather variable and outlook of given set of days function

def plot_composites(outlooks, weather, days, var_to_plot, bias_variable, bias_sign, hazard=None, centered=False, level=None, tod=0):
    """
    Plot composites of either an outlook probability field or a weather variable.

    Parameters
    ----------
    outlooks : xr.Dataset
        Contains prob(time, y, x, hazard), lat(y,x), lon(y,x).
    weather : xr.Dataset
        Contains various gridded weather variables with day as a dimension.
    days : list-like
        Subset of days (datetime64 or strings) to average over.
    var_to_plot : str
        Name of the variable to composite and plot (in either `weather` or `outlooks`).
    hazard : str or None, optional
        If plotting outlook probabilities, specify which hazard ('Wind', 'Hail', etc.).
    centered : bool, optional
        If True, recenters composites on max prob location (stubbed for now).
    level : int or None, optional
        Pressure level to select (default = 500 for level-dependent variables).
    tod : int, optional
        Time of day index to select. Defaults to 0.
    """

    if centered:
        raise NotImplementedError("Centered composites not implemented yet.")

    # --- 1. Outlook case ---
    if var_to_plot == "prob" or var_to_plot in outlooks.data_vars:
        if hazard is None:
            raise ValueError("Must specify a hazard when plotting outlook probabilities.")
        sub = outlooks.sel(time=days, hazard=hazard)
        comp = sub["prob"].mean(dim="time")

        lats = outlooks["lat"]
        lons = outlooks["lon"]
        title = f"Composite {var_to_plot} field when {hazard} {bias_variable} is {bias_sign}"

    # --- 2. Weather case ---
    elif var_to_plot in weather.data_vars:
        sub = weather[var_to_plot]

        # Select days
        if "day" in sub.dims:
            sub = sub.sel(day=days)

        # Handle level (default 500 hPa if applicable)
        if "level" in sub.dims:
            use_level = 500 if level is None else level
            if use_level in sub["level"]:
                sub = sub.sel(level=use_level)
                lev_text = f", {use_level} hPa"
            else:
                raise ValueError(f"Requested level {use_level} not found in variable {var_to_plot}.")
        else:
            lev_text = ""

        # Handle tod (always select one, default 0)
        if "tod" in sub.dims:
            sub = sub.sel(tod=tod)

        # Average over days
        comp = sub.mean(dim="day")

        lats = weather.get("latitude", weather.get("lat"))
        lons = weather.get("longitude", weather.get("lon"))
        title = f"Composite {var_to_plot} anomoly field when {hazard} {bias_variable} is {bias_sign}"

    else:
        raise KeyError(f"Variable {var_to_plot} not found in outlooks or weather datasets.")

    # --- 3. Plot composite map ---
    fig, ax = plt.subplots(
        figsize=(9, 6),
        subplot_kw={"projection": ccrs.LambertConformal(central_longitude=-95, central_latitude=35)}
    )

    ax.set_extent([-125, -67, 25, 50], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.STATES, linewidth=0.7)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.7)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)

    if var_to_plot == "prob":
        cmap = colortables.get_colortable("NWSReflectivity")
        vmin = 0
        vmax = .15
        if hazard == "Tornado":
            vmax = .05
    else:
        cmap = plt.get_cmap("coolwarm")
        vmin = -1
        vmax = 1

    pcm = ax.pcolormesh(
        lons,
        lats,
        comp,
        vmin=vmin,
        vmax=vmax,
        cmap=cmap,
        transform=ccrs.PlateCarree()
    )

    plt.colorbar(pcm, ax=ax, orientation="horizontal", pad=0.05, label=var_to_plot)
    
    ax.set_title(title)

    # TODO: save properly, make bounds +- 1. 

    outdir = f'figs/composites/{var_to_plot}/'

    # Create it if it doesn't exist
    os.makedirs(outdir, exist_ok=True)

    # Save the figure
    plt.savefig(
        f'{outdir}{hazard}_{bias_variable}_{bias_sign}_composite.png',
        dpi=300,
        bbox_inches='tight'
    )

    #plt.show()
    plt.close()

    # TODO: handle wind barbs?, handle centering, save (optionally)

In [86]:
# plot single day function

In [87]:
outlooks = xr.open_dataset('data/raw_data/outlooks.nc')
weather = xr.open_zarr('data/processed_data/train_inputs_small.zarr').load()

In [88]:
for hazard in day_ds.hazard.values:
    for variable in day_ds.variable.values:
        for bias_sign in ['pos', 'neg', 'zero']:
            days = day_ds['time'][(day_ds.bias_sign.sel(hazard=hazard, variable=variable) == bias_sign).values].values
            
            if len(days) > 0:
                #print(days)
                #if hazard == "Wind" and variable == "north_shift":
                print(f'Hazard: {hazard}, Variable: {variable}, Bias Sign: {bias_sign}, Number of Days: {len(days)}')
                for var in ['prob']:
                    plot_composites(outlooks, weather, days, var, variable, bias_sign, hazard, centered=False) 
                    # TODO centered = true   

Hazard: All Hazard, Variable: bias, Bias Sign: pos, Number of Days: 122
Hazard: All Hazard, Variable: bias, Bias Sign: neg, Number of Days: 90
Hazard: All Hazard, Variable: bias, Bias Sign: zero, Number of Days: 120
Hazard: All Hazard, Variable: east_shift, Bias Sign: pos, Number of Days: 98
Hazard: All Hazard, Variable: east_shift, Bias Sign: neg, Number of Days: 104
Hazard: All Hazard, Variable: east_shift, Bias Sign: zero, Number of Days: 130
Hazard: All Hazard, Variable: north_shift, Bias Sign: pos, Number of Days: 102
Hazard: All Hazard, Variable: north_shift, Bias Sign: neg, Number of Days: 107
Hazard: All Hazard, Variable: north_shift, Bias Sign: zero, Number of Days: 123
Hazard: Wind, Variable: bias, Bias Sign: pos, Number of Days: 106
Hazard: Wind, Variable: bias, Bias Sign: neg, Number of Days: 90
Hazard: Wind, Variable: bias, Bias Sign: zero, Number of Days: 136
Hazard: Wind, Variable: east_shift, Bias Sign: pos, Number of Days: 105
Hazard: Wind, Variable: east_shift, Bias S

In [ ]:
# for a specific model, for a specific target variable, break days somehow--days where model predicts positive, negative, and near zero. Or where model error is +/-/0? Then cross with true biases, and composite each of those categories.

Only run once, don't need to again:

In [90]:
# plot composites and look at examples from 
outlooks = xr.open_dataset('data/raw_data/grid_outlooks.nc')
outlooks = outlooks.assign_coords(time=pd.to_datetime(outlooks.time.astype(str), format='%Y%m%d%H%M'))

# Select Day 1 outlooks (including hazard types)
outlooks = outlooks.sel(outlook=[o for o in outlooks.outlook.values if o.startswith('Day 1')])

# Select only times in known list
outlooks = outlooks.sel(time=outlooks.time.isin(day_ds['time']))

def map_outlook_to_hazard(o):
    """Map Day 1 outlook strings to clean hazard names."""
    if o == 'Day 1':
        return 'All Hazard'
    elif re.search(r'Wind', o):
        return 'Wind'
    elif re.search(r'Hail', o):
        return 'Hail'
    elif re.search(r'Tornado', o):
        return 'Tornado'
    else:
        return None  # unexpected case

# Apply mapping
hazards = [map_outlook_to_hazard(o) for o in outlooks.outlook.values]

# Assign new coordinate
outlooks = outlooks.assign_coords(hazard=('outlook', hazards))

# Drop the old 'outlook' coordinate and rename
outlooks = outlooks.swap_dims({'outlook': 'hazard'}).drop_vars('outlook')
outlooks.to_netcdf("data/raw_data/outlooks.nc")